# Topic 3b - Attention in PyTorch

**Barclays Customer Support Intelligence System | Day 1, Topic 3b (Capstone)**

## What you will build

In Topic 3a you implemented scaled dot product attention from scratch in NumPy.
In this notebook you will:
1. Port dot product attention to a PyTorch `nn.Module` (with autograd)
2. Implement scaled dot product attention in PyTorch from scratch
3. Verify your implementation against `torch.nn.functional.scaled_dot_product_attention`
4. Apply your attention module to a complaint triage task and visualise attention weights

## Capstone lab (Tier 3 - open-ended)

Implement `ScaledDotProductAttention(nn.Module)` - signature and docstring only.
No step-by-step scaffold. You know the math from Topic 3a.

## Learning objectives

1. Translate the NumPy attention implementation to an `nn.Module` with gradient support
2. Understand how PyTorch handles batch dimensions and broadcasting automatically
3. Implement scaled dot product attention without scaffolding (Tier 3)
4. Verify a custom PyTorch implementation against a reference library function
5. Interpret attention weight heatmaps over complaint tokens

In [ ]:
# Environment setup for SageMaker Studio
# All attention demos run in this kernel - no remote training jobs.

!pip install -q "sagemaker>=2.200.0,<3.0.0" \
    "numpy<2" \
    "matplotlib>=3.7.0" \
    "seaborn>=0.12.0"

import sagemaker
from sagemaker import get_execution_role
import boto3

sess = sagemaker.Session()
role = get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name

print(f"Role: {role}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random
import warnings

warnings.filterwarnings("ignore")

# Reproducibility
def set_seeds(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

# Device: SageMaker Studio default is CPU. This notebook runs fine on CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
print()

# Complaint-domain vocabulary for all demos in this notebook.
# These are the tokens we will visualise attention over throughout.
COMPLAINT_TOKENS = [
    "unauthorised", "charge", "account", "fraud",
    "refund", "dispute", "urgent", "branch"
]
print(f"Complaint vocabulary for demos: {COMPLAINT_TOKENS}")

## Bridging from Topic 3a

In Topic 3a you implemented this function in NumPy:

```python
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(d_k)
    attention_weights = softmax(scores, axis=-1)
    output = np.matmul(attention_weights, V)
    return output, attention_weights
```

In this notebook we port the SAME logic to PyTorch as an `nn.Module`.
The equations do not change. What changes:
- `np.matmul` -> `torch.matmul` (or the `@` operator)
- numpy softmax -> `F.softmax(..., dim=-1)`
- Autograd handles gradients automatically (no backprop code needed)
- GPU support is free (just move tensors to device)

Let us start with dot product attention (the unscaled version) as a warm-up.

In [ ]:
# Beat 1: PyTorch attention without scaling saturates at large d_k.
# The softmax gets stuck near 0 or 1 -> gradients vanish -> model does not learn.
#
# We demonstrate this with a concrete forward + backward pass.

def unscaled_attention_forward(Q, K, V):
    """
    Dot product attention WITHOUT the 1/sqrt(d_k) scaling.
    This is the naive version that fails at large d_k.
    """
    # Raw dot product scores: (batch, T_q, d_k) x (batch, d_k, T_k) -> (batch, T_q, T_k)
    scores = torch.matmul(Q, K.transpose(-2, -1))
    # Softmax over keys
    attention_weights = F.softmax(scores, dim=-1)
    # Weighted sum of values
    output = torch.matmul(attention_weights, V)
    return output, attention_weights

print("Gradient magnitude vs key dimension (unscaled attention)")
print(f"{'d_k':>8}  {'max score':>12}  {'max attn':>12}  {'Q grad norm':>14}")
print("-" * 55)

batch_size = 2
T_q = 5
T_k = 8

for d_k in [4, 16, 64, 256, 512]:
    set_seeds(42)
    Q = torch.randn(batch_size, T_q, d_k, requires_grad=True)
    K = torch.randn(batch_size, T_k, d_k)
    V = torch.randn(batch_size, T_k, d_k)

    output, attn = unscaled_attention_forward(Q, K, V)

    # Simulate a loss and compute gradients
    loss = output.sum()
    loss.backward()

    max_score  = float(torch.max(torch.abs(torch.matmul(Q.detach(), K.transpose(-2, -1)))).item())
    max_attn   = float(torch.max(attn.detach()).item())
    q_grad_norm = float(Q.grad.norm().item())

    print(f"{d_k:>8}  {max_score:>12.2f}  {max_attn:>12.6f}  {q_grad_norm:>14.6f}")
    Q.grad = None  # reset for next iteration

print()
print("At d_k=512: max attention weight approaches 1.0 (one token dominates).")
print("Q gradient norm collapses -> the query receives almost no learning signal.")
print("FIX: divide scores by sqrt(d_k) before softmax.")

## Section 1 - Dot Product Attention in PyTorch

The warm-up: implement dot product attention as an `nn.Module`.
This is the unscaled version. We will add scaling in Section 2.

Notice how PyTorch's autograd handles the backward pass automatically -
no need to write gradient code by hand.

<!-- DIAGRAM: Flowchart of scaled dot product attention: Q and K^T feed into a MatMul node; output divided by sqrt(d_k); Softmax over key dimension; result multiplied with V via MatMul to produce context output. Tensor shapes (batch, T_q, d_k), (batch, d_k, T_k), (batch, T_q, T_k), (batch, T_k, d_v), (batch, T_q, d_v) labelled at each arrow. The sqrt(d_k) scaling node highlighted in orange. -->

```mermaid
graph TD
    Q["Q  Query matrix\n[batch, T_q, d_k]"] --> matmul1["MatMul\nQ x K^T"]
    K["K  Key matrix\n[batch, T_k, d_k]"] -->|"transpose"| KT["K^T\n[batch, d_k, T_k]"]
    KT --> matmul1
    matmul1 --> raw["Raw scores\n[batch, T_q, T_k]"]
    raw --> scale["Divide by sqrt(d_k)\nstabilises gradients"]
    scale --> sm["Softmax over key dim\n[batch, T_q, T_k]\nrows sum to 1"]
    sm --> matmul2["MatMul\nweights x V"]
    V["V  Value matrix\n[batch, T_k, d_v]"] --> matmul2
    matmul2 --> out["Context output\n[batch, T_q, d_v]"]

    style scale fill:#fda,stroke:#c63
    style out fill:#dfd,stroke:#3a3
```

*Figure: The diagram shows the same computation you implemented in Topic 3a, now with
PyTorch tensor shapes. The `sqrt(d_k)` scaling (highlighted) is the only difference
from plain dot product attention.*

<!-- DIAGRAM: Flowchart of scaled dot product attention: Q and K^T feed into a MatMul node; output divided by sqrt(d_k); Softmax over key dimension; result multiplied with V via MatMul to produce context output. Tensor shapes (batch, T_q, d_k), (batch, d_k, T_k), (batch, T_q, T_k), (batch, T_k, d_v), (batch, T_q, d_v) labelled at each arrow. The sqrt(d_k) scaling node highlighted in orange. -->

```mermaid
graph TD
    Q["Q  Query matrix\n[batch, T_q, d_k]"] --> matmul1["MatMul\nQ x K^T"]
    K["K  Key matrix\n[batch, T_k, d_k]"] -->|"transpose"| KT["K^T\n[batch, d_k, T_k]"]
    KT --> matmul1
    matmul1 --> raw["Raw scores\n[batch, T_q, T_k]"]
    raw --> scale["Divide by sqrt(d_k)\nstabilises gradients"]
    scale --> sm["Softmax over key dim\n[batch, T_q, T_k]\nrows sum to 1"]
    sm --> matmul2["MatMul\nweights x V"]
    V["V  Value matrix\n[batch, T_k, d_v]"] --> matmul2
    matmul2 --> out["Context output\n[batch, T_q, d_v]"]

    style scale fill:#fda,stroke:#c63
    style out fill:#dfd,stroke:#3a3
```

*Figure: The diagram shows the same computation you implemented in Topic 3a, now with
PyTorch tensor shapes. The `sqrt(d_k)` scaling (highlighted) is the only difference
from plain dot product attention.*

## Lab 1 - Implement DotProductAttention nn.Module (Tier 1 - Guided)

**Time**: 15-20 minutes

### Situation
The Barclays complaints ML team uses a PyTorch model to route incoming complaints.
They need a reusable attention module that can be dropped into any encoder-decoder model.

### Task
Implement `DotProductAttention` as an `nn.Module` with the correct forward pass.
You have seen the full working version in the demo above. Now implement it yourself.

### Action
Complete the three stubs in the forward method below.

### Result
The verification cell will check:
1. Output shape is (batch, T_q, d_v)
2. Attention weights sum to 1 along the key dimension
3. Your output matches the reference implementation numerically
4. Gradients flow through your module (the whole point of using PyTorch)

In [ ]:
# Lab 1: Implement DotProductAttention from scratch.
# Complete the three stubs in the forward method.

class MyDotProductAttention(nn.Module):
    """
    Dot product attention module.

    Formula: Attention(Q, K, V) = softmax(Q K^T) V

    Args (forward):
        query: (batch, T_q, d_k)
        key:   (batch, T_k, d_k)
        value: (batch, T_k, d_v)

    Returns:
        context:           (batch, T_q, d_v)
        attention_weights: (batch, T_q, T_k)
    """

    def __init__(self):
        super().__init__()

    def forward(self, query, key, value):
        # Step 1: Compute raw dot product scores.
        # Shape: (batch, T_q, d_k) x (batch, d_k, T_k) -> (batch, T_q, T_k)
        scores = None  # YOUR CODE

        # Step 2: Softmax over key positions.
        # Use F.softmax with the correct dim argument.
        attention_weights = None  # YOUR CODE

        # Step 3: Weighted sum of values.
        # Shape: (batch, T_q, T_k) x (batch, T_k, d_v) -> (batch, T_q, d_v)
        context = None  # YOUR CODE

        return context, attention_weights

# Quick sanity check
set_seeds(42)
my_attn = MyDotProductAttention()
test_q = torch.randn(2, 5, 16)
test_k = torch.randn(2, 5, 16)
test_v = torch.randn(2, 5, 16)
try:
    test_ctx, test_w = my_attn(test_q, test_k, test_v)
    if test_ctx is not None:
        print(f"Output shape: {test_ctx.shape}  (expected: torch.Size([2, 5, 16]))")
    else:
        print("Return values are None - complete all three steps.")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Lab 1 Verification
set_seeds(42)
ref_attn = DotProductAttention()
my_attn_verify = MyDotProductAttention()

q_v = torch.randn(2, 8, 32)
k_v = torch.randn(2, 8, 32)
v_v = torch.randn(2, 8, 32)

ref_ctx, ref_w = ref_attn(q_v, k_v, v_v)

all_pass = True

try:
    my_ctx, my_w = my_attn_verify(q_v, k_v, v_v)

    if my_ctx is None or my_w is None:
        print("FAIL: Return values are None. Complete all three steps.")
        all_pass = False
    else:
        # Shape check
        if my_ctx.shape == ref_ctx.shape:
            print(f"PASS: Output shape {my_ctx.shape}")
        else:
            print(f"FAIL: Expected shape {ref_ctx.shape}, got {my_ctx.shape}")
            all_pass = False

        # Attention weights sum to 1
        row_sums = my_w.sum(dim=-1)
        if torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5):
            print("PASS: Attention weights sum to 1 along key dimension")
        else:
            print(f"FAIL: Row sums: {row_sums[0].detach()}")
            all_pass = False

        # Numerical match
        if torch.allclose(my_ctx, ref_ctx, atol=1e-5):
            print("PASS: Output matches reference implementation")
        else:
            max_diff = float((my_ctx - ref_ctx).abs().max().item())
            print(f"FAIL: Max difference from reference: {max_diff:.6f}")
            all_pass = False

        # Gradient flow check
        q_grad_test = q_v.clone().requires_grad_(True)
        my_ctx2, _ = my_attn_verify(q_grad_test, k_v, v_v)
        my_ctx2.sum().backward()
        if q_grad_test.grad is not None:
            print("PASS: Gradients flow through the attention module")
        else:
            print("FAIL: No gradient on query - check your implementation")
            all_pass = False

except Exception as e:
    print(f"FAIL: Exception during forward pass: {e}")
    all_pass = False

if all_pass:
    print()
    print("All checks passed. DotProductAttention is correctly implemented.")

In [ ]:
# Lab 1 safety-net: run this if you did not finish Lab 1.
# SKIP this cell if you DID finish Lab 1.
if 'MyDotProductAttention' not in dir():
    MyDotProductAttention = DotProductAttention
    print("Using Lab 1 safety-net so the rest of the notebook can run.")
else:
    try:
        _test_q = torch.randn(1, 4, 8)
        _test_ctx, _ = MyDotProductAttention()(_test_q, _test_q, _test_q)
        if _test_ctx is None:
            MyDotProductAttention = DotProductAttention
            print("Using Lab 1 safety-net (implementation incomplete).")
    except Exception:
        MyDotProductAttention = DotProductAttention
        print("Using Lab 1 safety-net (implementation raised an error).")

### Stretch (fast finishers)

Add an optional `mask` parameter to your `MyDotProductAttention.forward` method.
A mask is a boolean tensor of shape `(batch, T_q, T_k)` where `True` means "ignore
this position". Apply the mask by setting masked positions to `-inf` before softmax.

This is used in decoders (causal masking) to prevent attending to future tokens.

```python
# Stretch: masked attention
# def forward(self, query, key, value, mask=None):
#     scores = ...
#     if mask is not None:
#         scores = scores.masked_fill(mask, float("-inf"))
#     attention_weights = F.softmax(scores, dim=-1)
#     context = ...
#     return context, attention_weights
```

### Homework Extension

Compare your `MyDotProductAttention` output to `F.scaled_dot_product_attention`
(PyTorch 2.0+ built-in). Note: you will need to adjust for the scaling factor.

```python
# Homework: Compare with PyTorch built-in
# output_builtin = F.scaled_dot_product_attention(q, k, v, scale=1.0)  # scale=1.0 disables scaling
# output_mine, _ = MyDotProductAttention()(q, k, v)
# print(torch.allclose(output_builtin, output_mine, atol=1e-4))
```

## Section 2 - Scaled Dot Product Attention in PyTorch

You implemented the unscaled version. Now we add the `sqrt(d_k)` scaling.

This is the EXACT operation at the core of every Transformer model.
After this section you will implement it yourself as the Tier 3 capstone.

In [ ]:
# Beat 1 resolved: scaling fixes the gradient saturation from Section 0.
# Side-by-side comparison of gradient norms: unscaled vs scaled at d_k=512.

def scaled_attention_forward(Q, K, V):
    """Scaled dot product attention with 1/sqrt(d_k) normalisation."""
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    attention_weights = F.softmax(scores, dim=-1)
    context = torch.matmul(attention_weights, V)
    return context, attention_weights

print("Unscaled vs Scaled attention - gradient norms at d_k=512")
print(f"{'version':>15}  {'max score':>12}  {'max attn':>12}  {'Q grad norm':>14}")
print("-" * 60)

d_k = 512
batch_size = 2
T_q = 5
T_k = 8

for label, forward_fn in [("Unscaled", unscaled_attention_forward),
                           ("Scaled", scaled_attention_forward)]:
    set_seeds(42)
    Q = torch.randn(batch_size, T_q, d_k, requires_grad=True)
    K = torch.randn(batch_size, T_k, d_k)
    V = torch.randn(batch_size, T_k, d_k)

    output, attn = forward_fn(Q, K, V)
    output.sum().backward()

    max_score   = float(torch.max(torch.abs(torch.matmul(Q.detach(), K.transpose(-2, -1)))).item())
    max_attn    = float(torch.max(attn.detach()).item())
    q_grad_norm = float(Q.grad.norm().item())

    print(f"{label:>15}  {max_score:>12.2f}  {max_attn:>12.6f}  {q_grad_norm:>14.6f}")

print()
print("Scaled version: smaller max attention weight, much larger gradient norm.")
print("The model can learn from both tokens now - scaling prevents one token from dominating.")

In [ ]:
# Beat 3: Reference implementation of scaled dot product attention.
# This is what you will implement yourself in the Tier 3 capstone lab.
# Study this carefully before starting the capstone.
#
# Instructor: walk through EVERY line. Shapes at each step.

class ScaledDotProductAttentionReference(nn.Module):
    """
    Reference implementation of scaled dot product attention.

    Formula: Attention(Q, K, V) = softmax( Q K^T / sqrt(d_k) ) V

    This is the fundamental operation of the Transformer (Vaswani et al., 2017).
    """

    def __init__(self, dropout_p=0.0):
        """
        Args:
            dropout_p: dropout probability on attention weights (0 = no dropout).
                       Dropout on attention weights regularises the model during training
                       by randomly zeroing out some attention connections.
        """
        super().__init__()
        # Optional dropout on attention weights
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, query, key, value):
        """
        Args:
            query: (batch, T_q, d_k)
            key:   (batch, T_k, d_k)
            value: (batch, T_k, d_v)

        Returns:
            output:            (batch, T_q, d_v)
            attention_weights: (batch, T_q, T_k)
        """
        # d_k is the key/query dimension - we scale by its square root
        d_k = query.shape[-1]                                        # scalar

        # Step 1: Scaled dot product scores
        # query @ key^T: (batch, T_q, d_k) x (batch, d_k, T_k) -> (batch, T_q, T_k)
        scores = torch.matmul(query, key.transpose(-2, -1))          # (batch, T_q, T_k)
        scores = scores / (d_k ** 0.5)                               # divide by sqrt(d_k)

        # Step 2: Softmax -> attention weights
        # dim=-1 means we softmax over the key dimension (T_k)
        # Each query position gets weights over ALL key positions that sum to 1
        attention_weights = F.softmax(scores, dim=-1)                # (batch, T_q, T_k)
        attention_weights = self.dropout(attention_weights)          # optional regularisation

        # Step 3: Weighted sum of values
        # (batch, T_q, T_k) x (batch, T_k, d_v) -> (batch, T_q, d_v)
        output = torch.matmul(attention_weights, value)              # (batch, T_q, d_v)

        return output, attention_weights

# --- Demo: complaint triage self-attention ---
set_seeds(42)

batch_size = 2
T_seq = len(COMPLAINT_TOKENS)  # 8 complaint tokens
d_k = 64
d_v = 64

# Simulate complaint embeddings (Q = K = V for self-attention)
Q_complaint = torch.randn(batch_size, T_seq, d_k)
K_complaint = Q_complaint   # self-attention: query = key
V_complaint = Q_complaint   # self-attention: value = key

ref_module = ScaledDotProductAttentionReference(dropout_p=0.0)
ref_output, ref_attn_weights = ref_module(Q_complaint, K_complaint, V_complaint)

print("ScaledDotProductAttention (Reference) - Complaint Self-Attention Demo")
print("=" * 60)
print(f"Q shape: {Q_complaint.shape}  -> (batch=2, tokens=8, d_k=64)")
print(f"K shape: {K_complaint.shape}")
print(f"V shape: {V_complaint.shape}")
print(f"Output shape:           {ref_output.shape}  -> (batch=2, tokens=8, d_v=64)")
print(f"Attention weights shape: {ref_attn_weights.shape}  -> (batch=2, 8 queries, 8 keys)")
print()
row_sums = ref_attn_weights[0].sum(dim=-1).detach()
print(f"Row sums of attn_weights[0]: {row_sums.numpy().round(4)}")
if torch.allclose(row_sums, torch.ones(T_seq), atol=1e-5):
    print("All must be 1.0 - confirmed.")
else:
    print("WARNING: rows do not sum to 1!")
print()

# Compare with PyTorch built-in F.scaled_dot_product_attention (PyTorch 2.0+)
try:
    builtin_output = F.scaled_dot_product_attention(Q_complaint, K_complaint, V_complaint)
    match = torch.allclose(ref_output, builtin_output, atol=1e-4)
    print(f"Matches F.scaled_dot_product_attention: {match}")
    print("Our reference implementation agrees with PyTorch's built-in function.")
except AttributeError:
    print("F.scaled_dot_product_attention requires PyTorch 2.0+. Skipping comparison.")

# Visualise attention weights (batch item 0)
plt.figure(figsize=(10, 8))
sns.heatmap(
    ref_attn_weights[0].detach().numpy(),
    xticklabels=COMPLAINT_TOKENS,
    yticklabels=COMPLAINT_TOKENS,
    cmap="Blues",
    annot=True,
    fmt=".2f",
    linewidths=0.5
)
plt.title("Self-Attention Weights - Complaint Tokens (random, untrained)")
plt.xlabel("Key tokens (what is being attended to)")
plt.ylabel("Query tokens (what is asking the question)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()
print("With random weights, attention is approximately uniform.")
print("After training on complaint-label pairs, high-severity tokens would")
print("attend strongly to each other: 'fraud' <-> 'unauthorised', 'charge' <-> 'refund'.")

<!-- DIAGRAM: Example 8x8 attention weight heatmap over complaint tokens. Rows are query tokens (unauthorised, charge, account, fraud, refund, dispute, urgent, branch); columns are key tokens (same). Show a TRAINED attention pattern where fraud has high attention weight on unauthorised and charge; refund attends to dispute; the diagonal (self-attention) is moderate. Annotate the high-weight cells to show the semantic clustering. Contrast this with a random/diagonal-dominated heatmap to show what training achieves. -->

```mermaid
graph TD
    subgraph trained ["Trained attention: semantic clusters emerge"]
        fraud["fraud"] -->|"high weight"| unauth["unauthorised"]
        fraud -->|"high weight"| charge["charge"]
        refund["refund"] -->|"high weight"| dispute["dispute"]
        urgent["urgent"] -->|"moderate"| fraud
        account["account"] -->|"low weight"| charge
    end

    subgraph random ["Untrained attention: diagonal dominant"]
        t1["token_1"] -->|"self only"| t1s["token_1"]
        t2["token_2"] -->|"self only"| t2s["token_2"]
    end

    note["Training teaches the model\nwhich tokens are semantically related.\nFraud attends to unauthorised and charge.\nRefund attends to dispute."]

    style fraud fill:#fda,stroke:#c33
    style refund fill:#ddf,stroke:#33c
    style note fill:#fff,stroke:#999
```

*Figure: A TRAINED attention heatmap is NOT diagonal. The model learns that "fraud" should
attend to "unauthorised" and "charge", and that "refund" is semantically related
to "dispute". This semantic clustering emerges from training on labelled complaints.*

### Discussion (3 minutes)

You have now seen `DotProductAttention` and `ScaledDotProductAttention` as `nn.Module`s.

1. The reference implementation has a `dropout_p` parameter on the attention weights.
   Why would you apply dropout specifically to attention weights rather than, say,
   to the output context vectors? What does "dropping out" an attention connection mean
   in terms of what the model is forced to do?

2. In Topic 3a we used word2vec embeddings as the Q, K, V inputs directly.
   In a real Transformer, Q, K, V are created by projecting the input embeddings
   through three separate learned weight matrices (W_Q, W_K, W_V).
   Why have separate projections? What would break if W_Q = W_K = W_V = the identity matrix?

3. The capstone asks you to implement `ScaledDotProductAttention` from scratch.
   Without looking at the reference above, how would you start?
   What is the first line of code you would write in `forward`?

## Capstone Lab - Implement Scaled Dot Product Attention from Scratch (Tier 3 - Open-Ended)

**Time**: 25-35 minutes | **Tier**: 3 (open-ended - function signature + docstring only)

This is the Day 1 capstone. You know the math from Topic 3a. You have studied the
reference implementation above. Now implement it yourself with NO scaffold.

### Situation
The Barclays ML platform team needs a production-quality attention module that:
- Works with batched complaint sequences
- Supports variable sequence lengths
- Returns both context vectors and attention weights (for interpretability)
- Passes a numerical verification against PyTorch's built-in function

### Task
Implement `ScaledDotProductAttention(nn.Module)`. The signature and docstring are
provided below. Do not look at the reference while implementing.

### Action
Complete the implementation below. No numbered steps. No `# YOUR CODE` placeholders.
You decide the structure.

### Result
The verification cell will check:
1. Output shape matches expected
2. Attention weights sum to 1 along the key dimension
3. Numerical match with `ScaledDotProductAttentionReference` from above
4. Gradient flows through the module

**Stretch**: After passing verification, add a boolean `mask` parameter that allows
masking out padding tokens (set masked scores to `-inf` before softmax). Test it.

**Homework Extension**: Verify your implementation matches
`F.scaled_dot_product_attention(Q, K, V)` (PyTorch 2.0+). They should agree to
within floating-point tolerance. If they differ, find the bug.

In [ ]:
# Capstone Lab: Implement scaled dot product attention.
# Signature and docstring provided. Implementation is yours.

class ScaledDotProductAttention(nn.Module):
    """
    Scaled dot product attention module.

    Implements: Attention(Q, K, V) = softmax( Q K^T / sqrt(d_k) ) V

    Parameters
    ----------
    dropout_p : float
        Dropout probability applied to attention weights during training.
        Default 0.0 (no dropout).

    Inputs (forward method)
    -----------------------
    query : torch.Tensor, shape (batch, T_q, d_k)
    key   : torch.Tensor, shape (batch, T_k, d_k)
    value : torch.Tensor, shape (batch, T_k, d_v)

    Returns
    -------
    output            : torch.Tensor, shape (batch, T_q, d_v)
    attention_weights : torch.Tensor, shape (batch, T_q, T_k)
                        Each row sums to 1.0 along the last dimension.
    """
    pass

In [ ]:
# Capstone Verification - run after completing ScaledDotProductAttention above.

set_seeds(42)

# Test data
batch_size = 3
T_q = 6
T_k = 8
d_k = 64
d_v = 64

q_test = torch.randn(batch_size, T_q, d_k)
k_test = torch.randn(batch_size, T_k, d_k)
v_test = torch.randn(batch_size, T_k, d_v)

# Reference
ref = ScaledDotProductAttentionReference(dropout_p=0.0)
ref_out, ref_w = ref(q_test, k_test, v_test)

all_pass = True

try:
    student_module = ScaledDotProductAttention(dropout_p=0.0)
    student_out, student_w = student_module(q_test, k_test, v_test)

    if student_out is None or student_w is None:
        print("FAIL: Module returned None. Implement the forward method.")
        all_pass = False
    else:
        # Check 1: Output shape
        if student_out.shape == ref_out.shape:
            print(f"PASS: Output shape {student_out.shape}")
        else:
            print(f"FAIL: Expected output shape {ref_out.shape}, got {student_out.shape}")
            all_pass = False

        # Check 2: Attention weights shape
        if student_w.shape == ref_w.shape:
            print(f"PASS: Attention weights shape {student_w.shape}")
        else:
            print(f"FAIL: Expected weights shape {ref_w.shape}, got {student_w.shape}")
            all_pass = False

        # Check 3: Weights sum to 1
        row_sums = student_w.sum(dim=-1)
        if torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5):
            print("PASS: Attention weights sum to 1 along key dimension")
        else:
            print(f"FAIL: Row sums min={row_sums.min():.4f} max={row_sums.max():.4f}")
            all_pass = False

        # Check 4: Numerical match with reference
        if torch.allclose(student_out, ref_out, atol=1e-5):
            print("PASS: Output matches ScaledDotProductAttentionReference")
        else:
            max_diff = float((student_out - ref_out).abs().max().item())
            print(f"FAIL: Max difference from reference: {max_diff:.6f}")
            print("Hint: check the scaling factor and the softmax dim.")
            all_pass = False

        # Check 5: Gradient flow
        q_grad = q_test.clone().requires_grad_(True)
        s_out, _ = student_module(q_grad, k_test, v_test)
        s_out.sum().backward()
        if q_grad.grad is not None:
            print("PASS: Gradients flow through ScaledDotProductAttention")
        else:
            print("FAIL: No gradient on query tensor")
            all_pass = False

        # Bonus: compare with PyTorch built-in
        try:
            builtin_out = F.scaled_dot_product_attention(q_test, k_test, v_test)
            builtin_match = torch.allclose(student_out, builtin_out, atol=1e-4)
            status = "PASS" if builtin_match else "FAIL"
            print(f"{status}: Matches F.scaled_dot_product_attention (PyTorch built-in)")
        except AttributeError:
            print("INFO: F.scaled_dot_product_attention not available (PyTorch < 2.0)")

except NotImplementedError:
    print("FAIL: Module raises NotImplementedError - remove the pass and implement forward()")
    all_pass = False
except Exception as e:
    print(f"FAIL: Exception during forward pass: {type(e).__name__}: {e}")
    all_pass = False

if all_pass:
    print()
    print("All capstone checks passed. Excellent work.")
    print("You have implemented the core operation of the Transformer from scratch in PyTorch.")

In [ ]:
# Capstone safety-net: run this if you did not finish the capstone.
# SKIP this cell if you DID finish the capstone.
_need_safety_net = False
try:
    _m = ScaledDotProductAttention(dropout_p=0.0)
    _q = torch.randn(1, 4, 16)
    _out, _w = _m(_q, _q, _q)
    if _out is None:
        _need_safety_net = True
except Exception:
    _need_safety_net = True

if _need_safety_net:
    print("Using capstone safety-net so the rest of the notebook can run.")
    ScaledDotProductAttention = ScaledDotProductAttentionReference

## Section 3 - Applying Your Attention Module to Complaint Triage

You have a working `ScaledDotProductAttention` module.
Let us use it in a minimal complaint triage model and visualise the learned attention pattern.

This is NOT a trained model - we use structured embeddings to simulate semantic proximity.
But the architecture shows how attention would fit into a production complaints routing system.

In [ ]:
# Applied demo: complaint triage self-attention with interpretability visualisation.
# We use the COMPLAINT_TOKENS from the imports cell throughout.
#
# A real system would load actual complaint embeddings from the model's embedding layer.
# Here we simulate plausible embeddings where semantically related tokens are closer.

set_seeds(42)

n_tokens = len(COMPLAINT_TOKENS)  # 8
d_model = 64

# Simulate complaint embeddings with some structure:
# "unauthorised", "charge", "fraud" should be semantically close
# "refund", "dispute" should be close
# We add a small perturbation around two cluster centres.
cluster_fraud = torch.randn(d_model)
cluster_account = torch.randn(d_model)

def make_complaint_embeddings(cluster_fraud, cluster_account, d_model, seed=42):
    """Create plausible complaint token embeddings with semantic clustering."""
    torch.manual_seed(seed)
    embeddings = torch.zeros(n_tokens, d_model)
    # Fraud cluster: unauthorised (0), charge (1), fraud (3)
    for i in [0, 1, 3]:
        embeddings[i] = cluster_fraud + 0.3 * torch.randn(d_model)
    # Account/resolution cluster: account (2), refund (4), dispute (5)
    for i in [2, 4, 5]:
        embeddings[i] = cluster_account + 0.3 * torch.randn(d_model)
    # Standalone: urgent (6), branch (7)
    embeddings[6] = torch.randn(d_model)
    embeddings[7] = torch.randn(d_model)
    return embeddings

complaint_emb = make_complaint_embeddings(cluster_fraud, cluster_account, d_model)

# Add batch dimension: (1, 8, 64) - one complaint message, 8 tokens
Q_appl = complaint_emb.unsqueeze(0)   # (1, 8, 64)
K_appl = Q_appl                        # self-attention
V_appl = Q_appl

# Use the student's attention module (or safety-net)
appl_module = ScaledDotProductAttention(dropout_p=0.0)
appl_output, appl_attn_weights = appl_module(Q_appl, K_appl, V_appl)

print(f"Input complaint embeddings: {Q_appl.shape}")
print(f"Output context vectors:     {appl_output.shape}")
print(f"Attention weights:          {appl_attn_weights.shape}")
print()

# Visualise
attn_np = appl_attn_weights[0].detach().numpy()

plt.figure(figsize=(10, 8))
sns.heatmap(
    attn_np,
    xticklabels=COMPLAINT_TOKENS,
    yticklabels=COMPLAINT_TOKENS,
    cmap="Reds",
    annot=True,
    fmt=".3f",
    linewidths=0.5
)
plt.title("Self-Attention Weights - Complaint Triage\n(structured embeddings, no training)")
plt.xlabel("Key tokens (being attended to)")
plt.ylabel("Query tokens (asking the question)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

print("With structured embeddings (fraud cluster + account cluster):")
print("'unauthorised', 'charge', 'fraud' should attend to each other.")
print("'account', 'refund', 'dispute' should attend to each other.")
print()

# Verify clustering effect
fraud_tokens = [0, 1, 3]   # unauthorised, charge, fraud
acct_tokens  = [2, 4, 5]   # account, refund, dispute

avg_intra_fraud = float(attn_np[np.ix_(fraud_tokens, fraud_tokens)].mean())
avg_cross       = float(attn_np[np.ix_(fraud_tokens, acct_tokens)].mean())

print(f"Average attention within fraud cluster:    {avg_intra_fraud:.4f}")
print(f"Average attention fraud -> account cluster: {avg_cross:.4f}")
if avg_intra_fraud > avg_cross:
    print("Confirmed: fraud tokens attend more to each other than to account tokens.")
    print("This is what semantic clustering in embeddings produces.")
else:
    print("With random weights the clustering effect may not appear - training would sharpen this.")

### Discussion (3 minutes)

You have applied your `ScaledDotProductAttention` to a complaint triage scenario.

1. The heatmap shows attention weights over 8 complaint tokens. In a real deployment,
   a complaint might have 50-200 tokens. What happens to the attention weight matrix
   as sequence length grows? If a complaint has 100 tokens, how large is the attention
   matrix? What does this mean for memory and compute?

2. The clustering effect above (fraud tokens attending to each other) emerged from
   structured embeddings, not from training. In a deployed model, a fraud analyst
   wants to know WHY the model flagged a complaint as high-severity.
   How would you use the attention weight matrix to produce an explanation?
   What are the limitations of using attention as an explanation?

3. `nn.MultiheadAttention` runs 4-8 attention heads in parallel. If each head
   specialises in different relationships (one head for fraud keywords, another for
   account action keywords), what does the final output capture that a single-head
   model would miss?

## Section 4 - Multi-Head Attention: Reference Only

Multi-head attention runs several scaled dot product attentions in parallel,
each with its own Q, K, V projections, then concatenates the results.

You do not need to implement this today. The reference below shows how
`ScaledDotProductAttention` composes into `nn.MultiheadAttention`.
We will build the full multi-head attention module in Topic 4 (Transformers).

In [ ]:
# Reference: nn.MultiheadAttention using PyTorch's built-in module.
# We will implement multi-head attention from scratch in Topic 4 (Transformers).
# For now, verify it works and understand the parameter count.

set_seeds(42)

embed_dim = 64    # must be divisible by num_heads
num_heads = 4     # 4 parallel attention heads, each with d_k = embed_dim // num_heads = 16

mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads,
                             dropout=0.0, batch_first=True)

# batch_first=True means input is (batch, seq, embed_dim)
batch_size_mha = 2
T_seq_mha = len(COMPLAINT_TOKENS)   # 8

Q_mha = torch.randn(batch_size_mha, T_seq_mha, embed_dim)
K_mha = Q_mha
V_mha = Q_mha

attn_output_mha, attn_weights_mha = mha(Q_mha, K_mha, V_mha)

print("nn.MultiheadAttention reference demo")
print("=" * 40)
print(f"embed_dim={embed_dim}, num_heads={num_heads}, d_k_per_head={embed_dim//num_heads}")
print(f"Input shape:         {Q_mha.shape}")
print(f"Output shape:        {attn_output_mha.shape}")
print(f"Attention weights:   {attn_weights_mha.shape}  -> (batch, T_q, T_k) averaged across heads")
print()

# Parameter count: Q, K, V in_proj + out_proj
total_params_mha = sum(p.numel() for p in mha.parameters())
print(f"Total parameters in MultiheadAttention: {total_params_mha:,}")
print(f"Breakdown:")
print(f"  in_proj (Q+K+V projections):  3 x {embed_dim} x {embed_dim} = {3*embed_dim*embed_dim}")
print(f"  out_proj:                     {embed_dim} x {embed_dim} = {embed_dim*embed_dim}")
print(f"  biases:                       {4*embed_dim}")
print()
print(f"In Topic 4 you will build this from {num_heads} x ScaledDotProductAttention heads.")

## Wrap-Up

### What you built today (Day 1 summary)

| Topic | What you implemented |
|-------|---------------------|
| 3a    | Bahdanau (additive) attention in NumPy - alignment scores, weights, context vector |
| 3a    | Dot product attention in NumPy |
| 3a    | Scaled dot product attention in NumPy |
| 3b    | `DotProductAttention` as `nn.Module` in PyTorch (Lab 1) |
| 3b    | `ScaledDotProductAttention` as `nn.Module` (Tier 3 capstone) |
| 3b    | Applied self-attention to complaint tokens with heatmap visualisation |

### Key principles to carry forward

1. The seq2seq bottleneck is fixed by attention: instead of one context vector,
   we compute a different weighted average of encoder states at each decoder step.

2. Scaled dot product attention (Vaswani et al., 2017) is the core operation of
   every modern Transformer model. The scaling by `1/sqrt(d_k)` prevents gradient
   saturation at large embedding dimensions.

3. Attention is largely parameter-free: the module itself has no weights.
   The learned weights live in the Q, K, V projection matrices outside the module.

4. Attention weights are interpretable: visualise them as a heatmap to understand
   what the model focuses on. This is critical for financial AI (explainability requirements).

### What is coming in Topic 4 - Transformers

In Topic 4 you will combine multiple `ScaledDotProductAttention` heads into
a full multi-head attention layer, add positional encoding, feed-forward blocks,
and build a complete Transformer encoder from scratch in PyTorch.
The capstone will be a GPU training job on a translation task.

## Homework Extensions

### Homework 1: Verify Against PyTorch Built-In

Your `ScaledDotProductAttention` should produce the same output as
`torch.nn.functional.scaled_dot_product_attention` (PyTorch 2.0+).

```python
# Homework 1: verify numerical match with built-in
import torch.nn.functional as F

q = torch.randn(2, 6, 32)
k = torch.randn(2, 6, 32)
v = torch.randn(2, 6, 32)

my_output, _ = ScaledDotProductAttention()(q, k, v)
builtin_output = F.scaled_dot_product_attention(q, k, v)

match = torch.allclose(my_output, builtin_output, atol=1e-4)
print(f"Match with F.scaled_dot_product_attention: {match}")
# Should print True. If not, check your scaling factor and softmax dim.
```

### Homework 2: Add Causal Masking

Extend your `ScaledDotProductAttention` to support causal (autoregressive) masking.
A causal mask prevents position i from attending to any position j > i.

Signature extension:

```python
def forward(self, query, key, value, causal=False):
    ...
    if causal:
        # Build upper-triangular mask: positions in the future
        # Masked positions should have score = -inf so softmax gives 0
        pass
    ...
```

Verification: `attention_weights[0]` should be lower-triangular (zeros above diagonal).
Test with a sequence of 6 tokens.

*End of Topic 3b - Attention in PyTorch*

End of Day 1. Next session: Topic 4 - Transformers + Translator Capstone.